In [1]:
import icechunk
import xarray as xr
from srm import catalog
import duckdb
import geopandas as gpd
import rasterix
import numpy as np
import pandas as pd
from rasterix.rasterize.rasterio import geometry_clip
import matplotlib.pyplot as plt
import seaborn as sns
from ibicus.debias import QuantileMapping
import xarray as xr
import s3fs
import numpy as np

In [2]:
def get_data(var_cesm = "TREFHT",
             var_era5 = "2m_temperature",
            unit_conv_cesm_to_era5 = 1):
    def fix_coords(ds: xr.Dataset):
        ds.coords["lon"] = (ds.coords["lon"] + 180) % 360 - 180
        ds = ds.sortby(ssp245.lon)
        return ds
    
    ssp245_cat = catalog.get("CESM2-WACCM-SSP245-icechunk")

    ssp245_storage = icechunk.s3_storage(
        bucket=ssp245_cat.bucket,
        prefix=ssp245_cat.prefix,
        from_env=True,
    )
    ssp245_repo = icechunk.Repository.open(ssp245_storage)
    ssp245_session = ssp245_repo.readonly_session("main")
    ssp245 = xr.open_zarr(ssp245_session.store, consolidated=False)
    ssp245 = fix_coords(ssp245)
    ssp245 = ssp245.proj.assign_crs(spatial_ref="epsg:4326")
    ssp245 = ssp245.isel(ensemble_member=0)
    
    model_historical_cat = catalog.get("CESM-WACCM-Historical-icechunk")
    
    model_historical_storage = icechunk.s3_storage(
        bucket=model_historical_cat.bucket,
        prefix=model_historical_cat.prefix,
        from_env=True,
    )
    model_historical_repo = icechunk.Repository.open(model_historical_storage)
    model_historical_session = model_historical_repo.readonly_session("main")
    model_historical = xr.open_zarr(model_historical_session.store, consolidated=False)
    model_historical = fix_coords(model_historical)
    
    g61pt5k_cat = catalog.get("CESM-WACCM-G6-1.5K-icechunk")
    
    g61pt5k_storage = icechunk.s3_storage(
        bucket=g61pt5k_cat.bucket,
        prefix=g61pt5k_cat.prefix,
        from_env=True,
    )
    g61pt5k_repo = icechunk.Repository.open(g61pt5k_storage)
    g61pt5k_session = g61pt5k_repo.readonly_session("main")
    g61pt5k = xr.open_zarr(g61pt5k_session.store, consolidated=False)
    g61pt5k = fix_coords(g61pt5k)
    g61pt5k = g61pt5k.isel(ensemble_member=0)
    
    era5_cat = catalog.get("ERA5")
    era5_storage = icechunk.s3_storage(
        bucket=era5_cat.bucket,
        prefix=era5_cat.prefix,
        from_env=True,
    )
    era5_repo = icechunk.Repository.open(era5_storage)
    era5_session = era5_repo.readonly_session("main")
    era5 = xr.open_zarr(era5_session.store).pipe(rasterix.assign_index)
    era5 = era5.proj.assign_crs(spatial_ref="epsg:4326")
    
    df = duckdb.sql(
        """install httpfs; load httpfs; install spatial; load spatial; SELECT name, ST_AsText(geom) as geometry FROM ST_Read('https://carbonplan-data.s3.us-west-2.amazonaws.com/countries-50m.json') WHERE NAME = 'South Africa'"""
    ).df()
    df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])
    south_africa_geom = gpd.GeoDataFrame(df, geometry="geometry")
    
    lon_min, lat_min, lon_max, lat_max = south_africa_geom.total_bounds
    lon_min = lon_min - 2
    lon_max = lon_max + 2
    lat_min = lat_min - 2
    lat_max = lat_max + 2
    
    era5_south_africa = geometry_clip(
        era5, south_africa_geom[["geometry"]], xdim="longitude", ydim="latitude"
    )
    
    era5_south_africa_bounds = era5.sel(
        longitude=slice(lon_min, lon_max),
        latitude=slice(lat_max, lat_min),  # Note: descending order for latitude
    )
    
    model_historical_south_africa_bounds = (
        model_historical.sortby("lat", ascending=False)
        .sel(
            lon=slice(lon_min, lon_max),
            lat=slice(lat_max, lat_min),  # Note: descending order for latitude
        )
        .load()
    )
    
    model_historical_south_africa = geometry_clip(
        model_historical.sortby("lat", ascending=False),
        south_africa_geom[["geometry"]],
        xdim="lon",
        ydim="lat",
    ).load()
    
    ssp245_south_africa_bounds = ssp245.sortby("lat", ascending=False).sel(
        lon=slice(lon_min, lon_max),
        lat=slice(lat_max, lat_min),  # Note: descending order for latitude
    )
    
    ssp245_south_africa = geometry_clip(
        ssp245.sortby("lat", ascending=False),
        south_africa_geom[["geometry"]],
        xdim="lon",
        ydim="lat",
    )
    
    g61pt5k_south_africa_bounds = g61pt5k.sortby("lat", ascending=False).sel(
        lon=slice(lon_min, lon_max),
        lat=slice(lat_max, lat_min),  # Note: descending order for latitude
    )
    
    g61pt5k_south_africa = geometry_clip(
        g61pt5k.sortby("lat", ascending=False),
        south_africa_geom[["geometry"]],
        xdim="lon",
        ydim="lat",
    )
    
    ds_future_dict = {
        "ssp245": ssp245_south_africa_bounds,
        "ssp245_clipped": ssp245_south_africa,
        "g61pt5k": g61pt5k_south_africa_bounds,
        "g61pt5k_clipped": g61pt5k_south_africa,
    }
    
    ds_hist_dict = {
        "model_hist_clipped": model_historical_south_africa,
        "model_hist": model_historical_south_africa_bounds,
        "era5_clipped": era5_south_africa,
        "era5": era5_south_africa_bounds,
    }
    
    for key, ds in ds_future_dict.items():
        ds = ds.where(ds["time.year"] >= 2050, drop=True)
        ds = ds.where(ds["time.year"] < 2070, drop=True)
        ds_future_dict[key] = ds[var_cesm] * unit_conv_cesm_to_era5
    
    for key, ds in ds_hist_dict.items():
        ds = ds.where(ds["time.year"] >= 1978, drop=True)
        ds = ds.where(ds["time.year"] < 2015, drop=True)
        if key in ["model_hist", "model_hist_clipped"]:
            ds_hist_dict[key] = ds[var_cesm] * unit_conv_cesm_to_era5
        else:
            ds_hist_dict[key] = ds[var_era5]
    
    dict_all = ds_future_dict | ds_hist_dict

    dict_all["era5_coarse"] = dict_all["era5"].interp(
        longitude=dict_all["ssp245"].lon,
        latitude=dict_all["ssp245"].lat,
        method="linear",
    )

    return dict_all

In [3]:
def debias_simulations(debiaser, dict_all):
    tas_cm_hist_debiased = debiaser.apply(
        obs=dict_all["era5_coarse"].values,
        cm_hist=dict_all["model_hist"].values,
        cm_future=dict_all["model_hist"].values,
        time_obs=dict_all["era5_coarse"]["time"].values,
        time_cm_hist=dict_all["model_hist"]["time"].values,
    )
    
    g61pt5k_fut_debiased = debiaser.apply(
        obs=dict_all["era5_coarse"].values,
        cm_hist=dict_all["model_hist"].values,
        cm_future=dict_all["g61pt5k"].values,
        time_obs=dict_all["era5_coarse"]["time"].values,
        time_cm_hist=dict_all["model_hist"]["time"].values,
        time_cm_future=dict_all["g61pt5k"]["time"].values,
    )
    
    ssp245_fut_debiased = debiaser.apply(
        obs=dict_all["era5_coarse"].values,
        cm_hist=dict_all["model_hist"].values,
        cm_future=dict_all["ssp245"].values,
        time_obs=dict_all["era5_coarse"]["time"].values,
        time_cm_hist=dict_all["model_hist"]["time"].values,
        time_cm_future=dict_all["ssp245"]["time"].values,
    )
    
    dict_all["model_hist_debiased"] = xr.DataArray(
        data=tas_cm_hist_debiased,
        coords={
            "lat": dict_all["model_hist"]["lat"],
            "lon": dict_all["model_hist"]["lon"],
            "time": dict_all["model_hist"]["time"],
        },
        dims=["time", "lat", "lon"],
    )
    
    dict_all["g61pt5k_debiased"] = xr.DataArray(
        data=g61pt5k_fut_debiased,
        coords={
            "lat": dict_all["g61pt5k"]["lat"],
            "lon": dict_all["g61pt5k"]["lon"],
            "time": dict_all["g61pt5k"]["time"],
        },
        dims=["time", "lat", "lon"],
    )
    
    
    dict_all["ssp245_debiased"] = xr.DataArray(
        data=ssp245_fut_debiased,
        coords={
            "lat": dict_all["ssp245"]["lat"],
            "lon": dict_all["ssp245"]["lon"],
            "time": dict_all["ssp245"]["time"],
        },
        dims=["time", "lat", "lon"],
    )

    return dict_all

In [4]:
def calculate_error_map(obs_coarse, obs_fine):

    def calculate_doy_means(ds):
        ds_xr = xr.DataArray(
            ds.data,
            dims=ds.dims,
            coords={k: v for k, v in ds.coords.items() if k != "spatial_ref"},
        )

        ds_xr = ds_xr.assign_coords(time=("time", pd.to_datetime(ds["time"].values)))

        ds_xr_doy_mean = ds_xr.groupby("time.dayofyear").mean("time")

        return ds_xr_doy_mean

    obs_coarse_reset = obs_coarse.reset_coords(["longitude", "latitude"], drop=True)
    obs_coarse_on_fine_grid = obs_coarse_reset.interp(
        lon=dict_all["era5"]["longitude"],
        lat=dict_all["era5"]["latitude"],
        method="linear",
    )
    error_map = obs_fine.mean(dim="time") - obs_coarse_on_fine_grid.mean(dim="time")

    obs_fine_doy_means = calculate_doy_means(obs_fine)

    obs_coarse_on_fine_grid_doy_means = calculate_doy_means(obs_coarse_on_fine_grid)

    error_map = obs_fine_doy_means - obs_coarse_on_fine_grid_doy_means

    return error_map

In [5]:
def downscale_from_coarse(da, error_map, fine_grid):
    da_fine_grid = da.interp(
        lon=fine_grid["longitude"],
        lat=fine_grid["latitude"],
        method="linear",
    )

    return da_fine_grid.groupby("time.dayofyear") + error_map

In [6]:
varname='tas'
algorithm="BCSD_parametric"
dict_all = get_data()
debiaser = QuantileMapping.from_variable(variable=varname, mapping_type="parametric")
dict_all = debias_simulations(debiaser, dict_all)

error_map = calculate_error_map(
    obs_coarse=dict_all["era5_coarse"], obs_fine=dict_all["era5"]
)

for scenario in ["ssp245","g61pt5k","model_hist"]:
    dict_all[scenario+"_debiased_downscaled"] = downscale_from_coarse(
    da=dict_all[scenario+"_debiased"], error_map=error_map, fine_grid=dict_all['era5']
)

100%|██████████| 620/620 [00:00<00:00, 1659.12it/s]


In [12]:
for var in ds.data_vars:
    print(var, ds[var].dtype, type(ds[var].dtype))
for coord in ds.coords:
    print(coord, ds[coord].dtype, type(ds[coord].dtype))

tas float64 <class 'numpy.dtypes.Float64DType'>
time object <class 'numpy.dtypes.ObjectDType'>
lon float64 <class 'numpy.dtypes.Float64DType'>
lat float64 <class 'numpy.dtypes.Float64DType'>
longitude float64 <class 'numpy.dtypes.Float64DType'>
spatial_ref int64 <class 'numpy.dtypes.Int64DType'>
latitude float64 <class 'numpy.dtypes.Float64DType'>
dayofyear int64 <class 'numpy.dtypes.Int64DType'>
ensemble_member object <class 'numpy.dtypes.ObjectDType'>


In [13]:
for scenario in ["ssp245","g61pt5k","model_hist"]:
    ds = dict_all[scenario+"_debiased_downscaled"].to_dataset(name=varname)
    ds['ensemble_member'] = ds['ensemble_member'].astype("object")
    fname=varname+"_"+scenario+"_"+algorithm+".nc"
    
    # Write NetCDF to local file
    ds.to_netcdf(fname)
    
    # Upload to S3 using s3fs
    fs = s3fs.S3FileSystem(anon=False)
    fs.put(fname, 's3://carbonplan-srm/output/example_SouthAfrica/'+fname)
    
    print(fname+" successfully written to S3!")

tas_ssp245_BCSD_parametric.nc successfully written to S3!
tas_g61pt5k_BCSD_parametric.nc successfully written to S3!
tas_model_hist_BCSD_parametric.nc successfully written to S3!


In [11]:



scenario='ssp245'
algorithm="BCSD_parametric"



NetCDF successfully written to S3!
